# Data Extraction - Azure AI Document Intelligence + Azure OpenAI GPT-4o

This sample demonstrates how to extract structured data from any document using Azure AI Document Intelligence and Azure OpenAI GPT models.

![Data Extraction](../../../images/extraction-document-intelligence-openai.png)

This is achieved by the following process:

- Analyze a document using Azure AI Document Intelligence's `prebuilt-layout` model to extract the structure as Markdown.
- Construct a system prompt that defines the instruction for extracting structured data from documents.
- Construct a user prompt that includes specific extraction instruction for the type of document, and the Markdown content of the document.
- Use the Azure OpenAI chat completions API with the GPT-4o model to generate a structured output from the content.

## Objectives

By the end of this sample, you will have learned how to:

- Convert a document to Markdown format using Azure AI Document Intelligence.
- Use prompt engineering techniques to instruct GPT-4o to extract structured data from a type of document.
- Use the [Structured Outputs feature](https://learn.microsoft.com/en-us/azure/ai-services/openai/how-to/structured-outputs?tabs=python-secure) to extract structured data from a document using Azure OpenAI's GPT-4o model.
- Use the analysis result from Azure AI Document Intelligence to determine the confidence of the extracted structured output.
- Use the [logprobs](https://learn.microsoft.com/en-us/azure/ai-services/openai/reference#request-body:~:text=False-,logprobs,-integer) parameter in an OpenAI request to determine the confidence of the extracted structured output.

## Setup

### Import modules

This sample takes advantage of the following Python dependencies:

- **azure-ai-documentintelligence** to interface with the Azure AI Document Intelligence API for analyzing documents.
- **openai** to interface with the Azure OpenAI chat completions API to generate structured extraction outputs using the GPT-4o model.
- **azure-identity** to securely authenticate with deployed Azure Services using Microsoft Entra ID credentials.

The following local modules are also used:

- **modules.app_settings** to access environment variables from the `.env` file.
- **modules.comparison** to compare the output of the extraction process with expected results.
- **modules.document_intelligence_confidence** to evaluate the confidence of the extraction process based on the extracted structured output and the analysis result from Azure AI Document Intelligence.
- **modules.document_processing_result** to store the results of the extraction process as a file.
- **modules.openai_confidence** to calculate the confidence of the classification process based on the `logprobs` response from the API request.
- **modules.invoice** to provide the expected structured output JSON schema for invoice documents.
- **modules.utils** `Stopwatch` to measure the end-to-end execution time for the classification process.

In [19]:
import sys
sys.path.append('../../') # Import local modules

from IPython.display import display, Markdown
import os
import pandas as pd
import json
from dotenv import dotenv_values
from azure.ai.documentintelligence import DocumentIntelligenceClient
from azure.ai.documentintelligence.models import AnalyzeResult, ContentFormat
from openai import AzureOpenAI
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
import json

from modules.app_settings import AppSettings
from modules.utils import Stopwatch
from modules.accuracy_evaluator import AccuracyEvaluator
from modules.comparison import get_extraction_comparison
from modules.confidence import merge_confidence_values
from modules.document_intelligence_confidence import evaluate_confidence as di_evaluate_confidence
from modules.openai_confidence import evaluate_confidence as oai_evaluate_confidence
from modules.invoice import Invoice
from modules.taxdoc import TaxDocument
from modules.document_processing_result import DataExtractionResult

### Configure the Azure services

To use Azure AI Document Intelligence and Azure OpenAI, their SDKs are used to create client instances using a deployed endpoint and authentication credentials.

For this sample, the credentials of the Azure CLI are used to authenticate with the deployed services.

In [20]:
# Set the working directory to the root of the repo
working_dir = os.path.abspath('../../../')
settings = AppSettings(dotenv_values(f"{working_dir}/.env"))

# Configure the default credential for accessing Azure services using Azure CLI credentials
credential = DefaultAzureCredential(
    exclude_workload_identity_credential=True,
    exclude_developer_cli_credential=True,
    exclude_environment_credential=True,
    exclude_managed_identity_credential=True,
    exclude_powershell_credential=True,
    exclude_shared_token_cache_credential=True,
    exclude_interactive_browser_credential=True
)

openai_token_provider = get_bearer_token_provider(credential, 'https://cognitiveservices.azure.com/.default')

openai_client = AzureOpenAI(
    azure_endpoint=settings.openai_endpoint,
    azure_ad_token_provider=openai_token_provider,
    api_version="2024-10-01-preview" # Requires the latest API version for structured outputs.
)

document_intelligence_client = DocumentIntelligenceClient(
    endpoint=settings.ai_services_endpoint,
    credential=credential
)

In [21]:
path = f"{working_dir}/samples/assets/pricing/"
pdf_files = [f for f in os.listdir(path) if f.endswith('.pdf')]
# metadata_fname = "taxform12.json" # Change this to the file you want to evaluate
# metadata_fpath = f"{path}{metadata_fname}"

# with open(metadata_fpath, "r") as f:
#     data = json.load(f)
    
# expected = TaxDocument(**data['expected'])
# pdf_fname = data['fname']
# pdf_fpath = f"{path}{pdf_fname}"

# tax_evaluator = AccuracyEvaluator(match_keys=['first_name', 'last_name'])

## Extract data from the document

The following code block executes the data extraction process using Azure AI Document Intelligence and Azure OpenAI's GPT-4o model.

It performs the following steps:

1. Get the document bytes from the provided file path. _Note: In this example, we are processing a local document, however, you can use any document storage location of your choice, such as Azure Blob Storage._
2. Use Azure AI Document Intelligence to analyze the structure of the document and convert it to Markdown format using the pre-built layout model.
3. Using Azure OpenAI's GPT-4o model and its [Structured Outputs feature](https://learn.microsoft.com/en-us/azure/ai-services/openai/how-to/structured-outputs?tabs=python-secure), extract a structured data transfer object (DTO) from the content of the Markdown.

In [22]:
with Stopwatch() as di_stopwatch:
    for pdf_file in pdf_files:
        pdf_file_path = os.path.join(path, pdf_file)
        with open(pdf_file_path, "rb") as f:
            poller = document_intelligence_client.begin_analyze_document(
                "prebuilt-layout",
                analyze_request=f,
                output_content_format=ContentFormat.MARKDOWN,
                content_type="application/pdf"
            )
        
        result: AnalyzeResult = poller.result()
        markdown = result.content

        # Write the markdown output to a file
        markdown_filename = f"{pdf_file.split('.')[0]}_md"
        with open(f"{working_dir}/samples/assets/pricing/{markdown_filename}.md", "w") as md_file:
            md_file.write(markdown)


In [23]:
markdown_files = [f for f in os.listdir(f"{working_dir}/samples/assets/pricing/") if f.endswith('_md.md')]

for markdown_file in markdown_files:
    with open(f"{working_dir}/samples/assets/pricing/{markdown_file}", "r") as md_file:
        markdown_content = md_file.read()

    with Stopwatch() as oai_stopwatch:
        completion = openai_client.chat.completions.create(
            model=settings.gpt4o_model_deployment_name,
            messages=[
                {
                    "role": "system",
                    "content": "You are an AI assistant who is an expert in understanding pricing addendum and respond to questions.",
                },
                {
                    "role": "user",
                    "content": f"""Answer the following questions from the pricing addendum. Provide your answers in a summarized json format. If you dont know the answer, provide null.
                    1. Vendor & Asset Classes: Within the Introductions section, locate the markdown or HTML table that contains the columns "Security Type", "Primary", "Secondary", and "Tertiary". Extract the entire table and convert it into a JSON format 

                    2. Market Close: Within the Exchange Traded Equities and Commoditiesd section, look for "Official Close" text return True if its present else return False. Client Region: Within the Introductions section, identify the client region. If 'US Commingled Trusts' is mentioned, return 'US'. Otherwise, return the mentioned region.

                    3. Extract the base and counter currencies, calculation frequencies, and contract types from the Foreign Exchange Rates section. Identify the base currency and list all the counter currencies mentioned. Additionally, capture the calculation frequencies such as 30 day, and the contract types e.g. Spot/Forward. Ensure to capture the details accurately. Return the output in JSON format 
                    

                    4. Data to be extracted might be available within an image/flowchart
                    Price Types:  Under the section 'Exchange Traded Equities and Commodities', check if any specific price type is mentioned, such as 'Close Volume', 'Bid-Ask Mean', 'Last Traded Price', or 'All Prices'. If all types are mentioned, the section should be 'All Prices'. Provide the output in json format
                   
                    Exception Prices: 
                    * Missing Price: Under the section 'Validating Valuations', check if there is any reference to 'Missing Price' or the phrase 'missing price'. If found, set the value to true; otherwise, the default value should be false.
                    * Zero Price : Under the section 'Validating Valuations', check if there is any reference to 'Zero Price' or the phrase 'zero price'. If found, set the value to true; otherwise, the default value should be false.
                    * Zero Volume : Under the section 'Validating Valuations', check if there is any reference to 'Zero Volume' or the phrase 'zero volume'. If found, set the value to true; otherwise, the default value should be false.
                    * Inter-Vendor Tolerance: Under the section 'Validating Valuations', check if there is any reference to 'Vendor to vendor comparison' or the phrase 'Vendor to vendor comparison'. If found, set the value to true; otherwise, the default value should be false.
                    * Historical Tolerance: Under the section 'Validating Valuations', check if there is any reference to 'Prior to Current Price Movement' or the phrase 'prior to current price movement'. If found, set the value to true; otherwise, the default value should be false.
                    * NAV Impact:  Under the section 'Validating Valuations', check if there is any reference to 'Share class NAV impact' or the phrase Share class NAV impact'. If found, set the value to true; otherwise, the default value should be false.

                    5. Data to be extracted might be available within an image/flowchart
                    Under the section 'Sourcing Valuations', identify if any specific price type is mentioned, such as 'MID', 'Close Volume', 'Bid-Ask Mean', or 'Last Traded Price'. Price types will be listed under each instrument type. You can identify the instrument type based on the section header (e.g., 'Exchange Traded Equities and Commodities' is of type EQ, 'Foreign Exchange Rates' is of type FX). Your task is to identify the instrument type and then the price codes under that instrument type.
                    Follow the rules below to convert Price Types to Price Codes, and always adhere to these rules to return the valid price codes:
                    CLOSE: Close Volume
                    BID: Bid-Ask Mean
                    ASK: Bid-Ask Mean
                    LAST: Last Traded Price
                    Provide the output in json format

                    6. Extract the pricing tolerance details from 'Analysis for Exception Prices' section from all flow-charts.
 
                    Sample Prompt Instructions (Data to be extracted might be available within an image/flowchart)
                    * Primary close vendor: In the 'Exchange Traded Equities' flowchart, check for the phrase 'Primary Vendor'. If it is present, set the value to true; otherwise, the default value should be false.
                    * Secondary close vendor: In the 'Exchange Traded Equities' flowchart, check for the phrase 'Secondary Vendor'. If it is present, set the value to true; otherwise, the default value should be false.
                    * Spread Movement: In the 'Exchange Traded Equities' flowchart, identify the spread movement between the primary vendor price and the secondary vendor price. Extract the exact value of the spread movement.
                    * Prior valuation:  In the 'Exchange Traded Equities' flowchart, check for the phrase 'price variance of prior'. If it is present, set the value to true; otherwise, the default value should be false. 
                    * Match Movement Lower Limit: In the 'Exchange Traded Equities' flowchart, identify the lower limit tolerance within the phrase 'Is price variance of prior XXpm price to current XXpm price > XX% or <-XX%?'. The lower limit is the number after the '>' sign. Your task is to extract this number along without the percentage sign.
                    * Match Movement Upper Limit: In the 'Exchange Traded Equities' flowchart, identify the upper limit tolerance within the phrase 'Is price variance of prior XXpm price to current XXpm price > XX% or <-XX%?'. The lower limit is the number after the '<' sign. Your task is to extract this number along without the percentage sign.

                    7. Extract the pricing tolerance details from 'Identification of Exception Prices' section from all flow-charts.
 
                    Sample Prompt Instructions (Data to be extracted might be available within an image/flowchart)
                    * NAV Impact Value: In the 'Exchange Traded Equities' flowchart, identify the NAV impact value from the phrase like 'NAV impact of prior to current price movement > XXX?' within the flowchart. Extract the numeric value after the '>' sign, your task is to extract this number  without the currency symbol.
                    * Prior valuation:  In the 'Exchange Traded Equities' flowchart, check for the phrase 'prior to current price movement'. If it is present, set the value to true; otherwise, the default value should be false. 
                    """,
                },
                {
                    "role": "user",
                    "content": markdown_content,
                }
            ],
            response_format={ "type": "json_object" },
            max_tokens=4096,
            temperature=0.1,
            top_p=0.1,
            logprobs=True # Enabled to determine the confidence of the response.
        )

    # Write the model output to a JSON file
    output_filename = f"{markdown_file.split('_md')[0]}_output.json"
    output_filepath = os.path.join(path, output_filename)

    with open(output_filepath, "w") as json_file:
        json.dump(completion.choices[0].message.content, json_file, indent=4)

In [24]:
# Displays the output of the Azure AI Document Intelligence pre-built layout analysis in Markdown format.
display(Markdown(markdown))

lValuation Policy Addendum
to the Pricing Review Committee Guidelines and Procedures

Security Pricing Group, CLIENTXX
November 30, 2024

<!-- PageBreak -->


# Introduction

The Fund Financial Services (FFS) Security Pricing team is responsible for sourcing individual
security prices from numerous third party market data vendors, validating those prices using a
rigorous set of exception criteria analysis, and distributing individual security prices to our
internal partners for final Net Asset Value (NAV) dissemination of all domestically domiciled
funds. The team is comprised of crew members that take a disciplined approach to analyzing
securities as defined in the Pricing Sourcebook and the Pricing Review Committee Guidelines
and Procedures. This document outlines valuation policies for sourcing and validating equity
securities, fixed income securities, derivatives, and foreign exchange spot and forward rates for
funds governed by the Investment Company Act of 1940 and US Commingled Trusts.


<table>
<tr>
<th>Security Type</th>
<th>Primary</th>
<th>Secondary</th>
<th>Tertiary</th>
</tr>
<tr>
<td>Municipal Bonds</td>
<td>Bloomberg</td>
<td>IDC</td>
<td>N/A</td>
</tr>
<tr>
<td>Corporate Bonds</td>
<td>Reuters</td>
<td>IDC</td>
<td>N/A</td>
</tr>
<tr>
<td>Money Market Securities</td>
<td>Reuters</td>
<td>Bloomberg</td>
<td>N/A</td>
</tr>
<tr>
<td>Non-USD Equities</td>
<td>IDC</td>
<td>Reuters</td>
<td>N/A</td>
</tr>
<tr>
<td>USD Equities</td>
<td>IDC</td>
<td>Reuters</td>
<td>Bloomberg</td>
</tr>
<tr>
<td>Foreign Currency FX Spot/Forward Rates</td>
<td>WMR</td>
<td>Bloomberg</td>
<td>N/A</td>
</tr>
<tr>
<td>Private Equities</td>
<td>External Source</td>
<td>N/A</td>
<td>N/A</td>
</tr>
<tr>
<td>Commodities</td>
<td>Bloomberg</td>
<td>N/A</td>
<td>N/A</td>
</tr>
<tr>
<td>Fair Value (Equity)</td>
<td>Markit</td>
<td>ITG</td>
<td>N/A</td>
</tr>
</table>


# Sourcing Valuations


## Straight-Through Pricing

Valuations for most equity and fixed income securities are entered into the accounting system
via automated data transmissions from predefined primary pricing vendors (see Appendix 1 for
a full list of vendor sources by asset class). Secondary and tertiary vendor prices are used to
analyze securities identified through the exception-based pricing review process for certain

<!-- PageBreak -->

asset classes, as outlined by the pricing hierarchy.


## Exchange Traded Equities and Commoditiesd

The straight-through pricing process require vendors of equity securities to provide prices
throughout the day in accordance with regional market hours which includes markets like
Riyadh Stock Exchange operating on non-standard week. Transmitted prices follow the pricing
hierarchy as illustrated below.


<figure>

Official close
with volume

Bid-ask
mean

2

Last Traded
available
price

1

3

</figure>


Official close and bid-ask mean of an equity security are considered accurate reflections of
market value. If there is no current day market input for a security, then LAST price must be
used.


## Foreign Exchange Rates

Spot, 30 day forward and 60 day forward are collected against USD. FX Rates are valued using
calculated MID. Counter currencies to be collected against USD are Euro, British Pounds, Indian
Rupee, Thai Baht, Australian Dollar, Canadian Dollar and United Arab Emirates Dirham. FX Rates
are collected at close of business US (16:30 EST).


## Debt Securities

Most debt securities are traded over the counter and not on an exchange. As a result, the
straight-through pricing process and the pricing hierarchy are based on evaluated bids.
Extensive analysis is conducted utilizing vendor-to-vendor comparisons, prior-to-current bid
price movement, broker quotes, trade data, security-versus-benchmark yield and spread
comparisons, and advisor commentary across various asset types.

There are some instances in which debt security prices are not received though a vendor price
transmission. Valuations for these securities can be obtained using alternative means (e.g.
extracted from a vendor's website, received via e-mail or FTP/SFTP from our vendor, or
calculated based on a decision made by the PRC.) These security prices are then manually
entered into the accounting system by the Security Pricing team.

<!-- PageBreak -->


## Fair Valuing of Securities

Funds governed by the Investment Company Act of 1940 and US Commingled Trusts must have
NAVs reflective of 4pm ET market activity. All securities that trade in foreign markets and close
before 4pm ET will be valued at a fair value price based on security-specific information
provided by pricing vendors. Third-party vendors value foreign securities using historical data
and regression analysis models that incorporate issue-specific, industry-specific, and market
specific valuation changes that occur between the local market and US market closings. The
analysis of valuation changes results in an adjustment factor for each foreign security. The
exceptions to this are non-US convertible securities. These securities are priced as of their local
market close time.


# Validating Valuations


## Identification of Exception Prices

Securities that are not straight-through priced via vendor transmissions are identified as
exceptions and require additional analysis for valuation. These securities are identified as
exceptions throughout the pricing process. Pricing valuation exceptions can be grouped into six
main categories for valuation of different asset types. A detailed exception identification
process by outlier criteria can be found in Appendix 4.


<figure>

1

2

3

Vendor to vendor
comparison

Prior to current price
movement

Share class NAV
impact

4

5

6

Yield/spread
movement

Zero / Missing Price

Zero volume

</figure>


# Analysis of Exception Prices

Securities that are identified as exceptions during the daily pricing process are further examined
by researching security-specific and industry-specific events and by using a secondary and

<!-- PageBreak -->

sometimes tertiary source. Additionally, prior and current day trade data is reviewed in
conjunction with commentary from XXX fund advisors and broker quotes. Based on a review of
all the above data, the FFS Security Pricing team will determine which vendor source provided
the best price for the current day and use that source as the appropriate valuation decision for
the day.


<figure>

All municipals

☒

☒

Is NAV impact of prior to
current price movement >
$.003?

Is price variance of prior 4pm
price to current 4pm price > 2%
or <- 2%?

Is spread change ≥ 15 and price
movement > .5% (prior 4pm
price to current 4pm price)?

Or

Or

Y

Y

Y

</figure>


Outlier

<!-- PageBreak -->


# Exchange Traded Equities


<figure>

Is NAV impact of prior to
current price movement >
$.003?

Is price variance of prior 4pm
price to current 4pm price > 2%
or <- 2%?

Is difference between primary
vendor and secondary vendor
price >=0.01%?

☒

☒

</figure>


Outlier


<figure>

All international debt securities at local close

☒

Is there a difference between min and max
vendor price >2%?

Is a minimum of two vendor prices
available?

Y
V
☒

Is the max class level impact between
min and max vendor price >.001

N

Y

Y
V
☒

☒

</figure>


Outlier
